# DWDS API Test

Simple test of the DWDSMini client functionality.

In [1]:
import sys
sys.path.append('../../src')

from src.data.dwds_corpus_client import DWDSClient

client = DWDSClient()

INFO:src.data.dwds_corpus_client:DWDSClient initialized with 0.6s delay, 30s timeout


## Test 1: Dictionary

In [2]:
dict_data = client.dictionary("Bank")
print(f"Dictionary entries: {len(dict_data) if dict_data else 0}")
if dict_data:
    print(f"Sample: {dict_data[0]}")

INFO:src.data.dwds_corpus_client:Fetching dictionary data for 'Bank'
INFO:src.data.dwds_corpus_client:✅ Successfully loaded 2 dictionary entries for 'Bank'


Dictionary entries: 2
Sample: {'input': 'Bank', 'wortart': 'Substantiv', 'url': 'https://www.dwds.de/wb/Bank#1', 'lemma': 'Bank'}


## Test 2: Corpus

In [7]:
corpus_data = client.corpus_csv("Golf", limit=5)
print(f"Corpus rows: {len(corpus_data) if corpus_data is not None else 0}")
if corpus_data is not None and not corpus_data.empty:
    print(f"Columns: {list(corpus_data.columns)}")
    print(f"Sample date: {corpus_data['Date'].iloc[0]}")
    print(f"Sample text: {corpus_data['Hit'].iloc[0][:50]}...")

INFO:src.data.dwds_corpus_client:Fetching corpus data for 'Golf' (limit=5)
INFO:src.data.dwds_corpus_client:✅ Successfully loaded 5 corpus contexts


Corpus rows: 5
Columns: ['No.', 'Date', 'Genre', 'Bibl', 'Hit']
Sample date: 1999-12-31
Sample text: Laßt euch nicht als Kanonenfutter für einen Krieg ...


## Test 3: Timeseries

In [4]:
ts_data = client.timeseries("Bank")
print(f"Timeseries data: {type(ts_data)}")
if ts_data:
    print(f"Keys: {list(ts_data.keys()) if isinstance(ts_data, dict) else 'Not a dict'}")
print("Done!")

INFO:src.data.dwds_corpus_client:Fetching timeseries data for 'Bank' from 1900
ERROR:src.data.dwds_corpus_client:Network error fetching timeseries for 'Bank': Expecting value: line 2 column 1 (char 1)


Timeseries data: <class 'NoneType'>
Done!


## Test 4: General

In [11]:
# Extract contexts with temporal data for "Bank"
target_word = "Bank"

print(f"=== Corpus Contexts for '{target_word}' ===\n")

# Get corpus data with temporal metadata (error handling in client)
corpus_data = client.corpus_csv(target_word, limit=20)

if corpus_data is not None and not corpus_data.empty:
    print(f"✅ Extracted {len(corpus_data)} contexts")
    print(f"📊 Columns: {list(corpus_data.columns)}")

    # Add year column for analysis
    corpus_data['Year'] = corpus_data['Date'].str[:4]

    # Show temporal coverage
    years = corpus_data['Year'].dropna()
    print(f"📅 Date range: {corpus_data['Date'].min()} to {corpus_data['Date'].max()}")
    print(f"📈 Year range: {years.min()} - {years.max()}")

    # Show sample contexts
    print(f"\n--- Sample Contexts ---")
    for i, row in corpus_data.head(3).iterrows():
        print(f"\nContext {i+1}:")
        print(f"  📅 Date: {row['Date']}")
        print(f"  📝 Genre: {row['Genre']}")
        print(f"  📖 Source: {row['Bibl']}")
        print(f"  📄 Text: {row['Hit'][:100]}...")
        print(f"  {'─' * 60}")

    # Basic statistics
    print(f"\n--- Temporal Distribution ---")
    year_counts = corpus_data['Year'].value_counts().sort_index()
    print(f"Contexts per decade:")
    for year in year_counts.index[:5]:
        if year and len(year) == 4:
            decade = f"{year[:3]}0s"
            count = year_counts[year]
            print(f"  {decade}: {count} contexts")
else:
    print(f"❌ No corpus data available for '{target_word}'")

print(f"\n💡 This corpus data provides:")
print(f"   - Real usage contexts from German texts")
print(f"   - Precise dates for diachronic analysis")
print(f"   - Genre and source information")
print(f"   - Perfect for training temporal WSD models")

=== Corpus Contexts for 'Bank' ===



INFO:src.data.dwds_corpus_client:Fetching corpus data for 'Bank' (limit=20)
INFO:src.data.dwds_corpus_client:✅ Successfully loaded 20 corpus contexts


✅ Extracted 20 contexts
📊 Columns: ['No.', 'Date', 'Genre', 'Bibl', 'Hit']
📅 Date range: 1999-12-31 to 1999-12-31
📈 Year range: 1999 - 1999

--- Sample Contexts ---

Context 1:
  📅 Date: 1999-12-31
  📝 Genre: Belletristik
  📖 Source: Degenhardt, Franz Josef: Für ewig und drei Tage, Berlin: Aufbau-Verl. 1999, S. 38
  📄 Text: Hawa setzte sich auf die um den tragenden Pfosten in der Mitte der Schlaganlage führende Bank....
  ────────────────────────────────────────────────────────────

Context 2:
  📅 Date: 1999-12-31
  📝 Genre: Belletristik
  📖 Source: Degenhardt, Franz Josef: Für ewig und drei Tage, Berlin: Aufbau-Verl. 1999, S. 45
  📄 Text: Sie saßen manchmal in der Küche beieinander oder auf der Bank unter der Terrasse, lachten, erzählten...
  ────────────────────────────────────────────────────────────

Context 3:
  📅 Date: 1999-12-31
  📝 Genre: Belletristik
  📖 Source: Degenhardt, Franz Josef: Für ewig und drei Tage, Berlin: Aufbau-Verl. 1999, S. 88
  📄 Text: Sie saßen nach dem Abend

__________

## Test 5 -  Frequency Analysis Over Time

Analyze word frequency trends to identify periods of semantic change.

In [ ]:
# Analyze frequency trends for multiple polysemous words
analysis_words = ["Bank", "Schloss", "Zelle"]

print("=== Frequency Analysis Over Time ===\n")

timeseries_data = {}

for word in analysis_words:
    print(f"📈 Analyzing '{word}'...")
    
    # Get timeseries data (error handling in client)
    ts_data = client.timeseries(word, start=1900)
    
    if ts_data:
        timeseries_data[word] = ts_data
        print(f"   ✅ Retrieved timeseries data: {type(ts_data)}")
        
        if isinstance(ts_data, dict):
            print(f"   📊 Data keys: {list(ts_data.keys())}")
        elif isinstance(ts_data, list) and ts_data:
            print(f"   📊 Data points: {len(ts_data)}")
            print(f"   📅 Sample: {ts_data[0] if ts_data else 'No data'}")
    else:
        print(f"   ❌ No timeseries data available")
    
    print()

# Note about visualization
print("💡 Frequency analysis reveals:")
print("   - Usage patterns over time")
print("   - Potential periods of semantic change")
print("   - Relative frequency compared to other words")
print("   - Foundation for diachronic WSD modeling")

# Save data for later analysis
if timeseries_data:
    print(f"\n💾 Frequency data collected for {len(timeseries_data)} words")
    print("   Ready for temporal modeling and visualization")